Way to vectorize computation

\begin{equation*}
U = U_{coulomb} + U_{hard} = \sum_{i<j}\frac{q_iq_j}{||\bold{r}_i - \bold{r}_j||} + \sum_{i<j}\frac{1}{||\bold{r}_i - \bold{r}_j||^8}
\end{equation*}

define the square distance matrix:

\begin{equation*}
D_{ij}^2 = ||\bold{r}_i - \bold{r}_j||^2 = \bold{r}_i \cdot \bold{r}_i + \bold{r}_j \cdot \bold{r}_j - 2 \bold{r}_i \cdot \bold{r}_j
\end{equation*}

\begin{equation*}
\Lambda^{(1)}_{ij} = 1/D_{ij}
\end{equation*}

\begin{equation*}
\Lambda^{(8)}_{ij} = 1/D_{ij}^8
\end{equation*}

Using this we can rewrite the total energy as a quadratic-form (put to 0 the diagonal of $\Lambda$ avoiding self interaction divergent terms):

\begin{equation*}
U = \frac{1}{2} \left( \bold{q} \cdot \Lambda^{(1)}\bold{q} + \bold{1}\cdot\Lambda^{(8)}\bold{1} \right) = \frac{1}{2} \left( \bold{q} \cdot \Lambda^{(1)}\bold{q} + ||\Lambda^{(8)}||_F \right)
\end{equation*}

which is super fast to compute with numpy or cupy. $|| \cdot||_F$ is the Frobenius norm.

In [1]:
import numpy as np
import time

In [2]:
SEED = 42
np.random.seed(SEED)

BOX_LENGHT = 1
SPACE_DIM = 3

In [3]:
particle_type = np.dtype([
    ('position', np.float64, (SPACE_DIM,)),
    ('velocity', np.float64, (SPACE_DIM,)),
    ('mass', np.float64),
    ('charge', np.float64)
])

In [4]:
def compute_distance_square_matrix(positions : np.array) -> np.array:
    
    r2 = np.sum(positions*positions, axis=1)
    D2 = r2[:,None] + r2[None,:] - 2*(positions @ positions.T)
    np.fill_diagonal(D2, np.inf)
    
    return D2

def compute_coulomb_energy(charges : np.array, D2 : np.array) -> np.float64:

    energy = 0.5 * np.sum((charges[:,None] * charges[None,:]) / np.sqrt(D2))
    
    return energy

def compute_lennar_jones_energy(D2 : np.array) -> np.float64:
    
    D8 = np.empty_like(D2)
    np.power(D2, 4, out=D8)
    energy = np.sum(1 / D8)

    return energy

def compute_potential_energy(particles : np.ndarray) -> np.float64:
    
    D2 = compute_distance_square_matrix(particles['position'])
    
    energy_coulomb = compute_coulomb_energy(particles['charge'], D2)
    energy_lennar = compute_lennar_jones_energy(D2)
    
    return energy_coulomb + energy_lennar   

In [8]:
N = [100, 1000, 1500, 2000, 3000, 4000,10000]

for N_PARTICLES in N:
    particles = np.zeros(N_PARTICLES, dtype=particle_type)

    particles['position'] = np.random.uniform(low=0,high=1,size=(N_PARTICLES,SPACE_DIM))
    particles['velocity'] = np.random.uniform(low=0,high=1,size=(N_PARTICLES,SPACE_DIM))
    particles['mass'] = 1
    particles['charge'] = np.sign(np.random.uniform(-1, 1, size=N_PARTICLES))
    
    start = time.time()
    compute_potential_energy(particles=particles)
    end = time.time()
    
    print(f"N_particles = {N_PARTICLES}")
    print(f"Time CPU: {(end - start)*1000:.0f} ms\n")

N_particles = 100
Time CPU: 2 ms

N_particles = 1000
Time CPU: 71 ms

N_particles = 1500
Time CPU: 124 ms

N_particles = 2000
Time CPU: 218 ms

N_particles = 3000
Time CPU: 499 ms

N_particles = 4000
Time CPU: 735 ms

N_particles = 10000
Time CPU: 4189 ms

